---
#**CAMADA DE REDE: ENDEREÇAMENTO IP E ROTEAMENTO DE DADOS**

####**Relatório Técnico Acadêmico | Redes 2026 | Colaboratory**
**Data: 16/04/2026**
<div>Protocolo IP | Roteamento | IPv4/IPv6 | Datagramas</div>

*Estudo sobre o endereçamento lógico e a entrega de pacotes através de saltos de rede.*

##**Introdução Teórica**

A Camada de Rede é a responsável pelo endereçamento lógico e pelo roteamento de pacotes (datagramas) em uma rede de computadores. Segundo Nunes (2017), o principal protocolo desta camada é o **IP (Internet Protocol)**, que fornece um serviço de entrega não confiável e sem conexão. O processo de **Roteamento** determina o melhor caminho que um pacote deve seguir entre a origem e o destino, atravessando diversos roteadores. Com a exaustão dos endereços **IPv4** (32 bits), a transição para o **IPv6** (128 bits) tornou-se essencial para suportar a crescente demanda de dispositivos conectados, garantindo a escalabilidade global da internet.

---
##**Objetivos**

 - **Identificar** os endereços IPv4 e IPv6 atribuídos às interfaces de rede do host.
 - **Analisar** a tabela de roteamento local para entender o caminho de saída dos dados (Gateway).
 - **Verificar** a conectividade com servidores DNS e o tempo de vida do pacote (TTL).
 - **Demonstrar** o papel dos roteadores na interconexão de redes heterogêneas.

---
###**1. Verificação de Protocolos IP**

As redes modernas operam em "Dual Stack". Vamos verificar as configurações de endereçamento de 32 e 128 bits da máquina virtual.

In [6]:
# Listando IPs na versão 4 e 6 da interface principal
print("--- Endereços IPv4 Ativos ---")
!ip -4 addr show eth0 | grep inet
print("\n--- Endereços IPv6 Ativos ---")
!ip -6 addr show eth0 | grep inet6

--- Endereços IPv4 Ativos ---
    inet 172.28.0.12/16 brd 172.28.255.255 scope global eth0

--- Endereços IPv6 Ativos ---


####**O que observar:**
<a>O teste identificou o endereço IPv4 `172.28.0.12`. A ausência de um endereço IPv6 global nesta interface reflete uma característica comum em infraestruturas de nuvem legadas ou específicas, onde a pilha dupla (Dual Stack) ainda não está totalmente provisionada. Conforme Nunes (2017), isso evidencia o desafio da transição global de protocolos, onde o IPv4 ainda é o padrão predominante para comunicação interna em muitos data centers.</a>

---
###**2. Análise de Rota Padrão (Gateway)**

O roteador é o dispositivo que decide o próximo passo do pacote. Vamos ver para onde a máquina envia os dados que não pertencem à rede local.

In [9]:
# Exibindo a tabela de rotas do kernel Linux
!ip route show

default via 172.28.0.1 dev eth0 
172.28.0.0/16 dev eth0 proto kernel scope link src 172.28.0.12 


####**O que observar:**
<a>A linha que começa com `default via` indica o endereço do roteador (Gateway). Todo pacote destinado à internet é encapsulado e enviado para este endereço, que então processa o roteamento externo.</a>

---
###**3. Integridade e Ciclo de Vida do Pacote**

O campo TTL (Time to Live) no cabeçalho IP evita que pacotes fiquem circulando infinitamente em loops de rede. Vamos observar o TTL em uma resposta.

In [8]:
# Usando curl para observar o comportamento da rede (substituto para análise de pacote)
!curl -v https://www.google.com 2>&1 | grep "Connected to"

* Connected to www.google.com (142.251.154.119) port 443 (#0)


####**O que observar:**
<a>A conexão bem-sucedida indica que o pacote atravessou diversos roteadores intermediários. Cada roteador decrementa o valor do TTL; se chegar a zero, o pacote é descartado, garantindo a saúde da rede.</a>

---
###**4. Mapeamento de Topologia Lógica**

Vamos mapear os "hops" (saltos) para entender como a camada de rede interconecta diferentes redes geográficas.

In [11]:
# Instalando ferramenta e rastreando a rota para analisar os saltos (hops)
!apt-get install traceroute -y > /dev/null
!traceroute -n google.com

traceroute to google.com (142.250.31.138), 30 hops max, 60 byte packets
 1  172.28.0.1  0.038 ms  0.008 ms  0.006 ms
 2  142.250.208.219  1.247 ms 216.239.42.7  1.474 ms *
 3  74.125.37.152  0.620 ms 72.14.239.164  1.746 ms 172.253.68.72  0.746 ms
 4  216.239.57.96  2.442 ms 74.125.37.159  1.354 ms *
 5  142.250.209.110  1.116 ms 216.239.46.31  1.100 ms 142.251.77.241  1.085 ms
 6  172.253.72.27  0.687 ms 172.253.72.71  1.571 ms 172.253.72.67  1.454 ms
 7  * * *
 8  * * *
 9  * * *
10  * * *
11  * * *
12  * * *
13  * 142.250.31.138  0.519 ms *


####**O que observar:**
<a>Cada linha representa um roteador na Camada de Rede realizando o encaminhamento do datagrama IP.
>**Nota Técnica:** É esperado que alguns saltos exibam asteriscos (`* * *`); isso ocorre porque o ambiente Cloud mascara parte da topologia interna e firewalls bloqueiam respostas ICMP/UDP por segurança. O teste é validado ao observar a progressão dos saltos até a saída da rede local para a WAN.</a>

---
##**Conclusão**

A exploração prática da Camada de Rede revelou a complexidade e a importância da padronização dos protocolos de endereçamento. Através dos testes realizados, validamos que o **Protocolo IP** atua como a espinha dorsal da comunicação, permitindo que datagramas atravessem fronteiras de redes heterogêneas por meio do roteamento. A observação de endereços IPv4 exclusivos e a identificação de saltos mascarados (hops) no `traceroute` evidenciam as nuances de segurança das infraestruturas modernas em nuvem. Para o profissional de ADS, o domínio desses conceitos é vital não apenas para o diagnóstico de falhas, mas para a arquitetura de sistemas escaláveis e seguros que dependem de uma entrega de pacotes eficiente e previsível em escala global.

---
##**Referências Bibliográficas**

NUNES, Sergio Eduardo. **Redes de computadores**. Londrina: Editora e Distribuidora Educacional S.A., 2017. 200 p.

FOROUZAN, Behrouz A. **Comunicação de Dados e Redes de Computadores**. 4. ed. Porto Alegre: AMGH, 2010.

TANENBAUM, Andrew S.; WETHERALL, David J. **Redes de Computadores**. 5. ed. São Paulo: Pearson Prentice Hall, 2011.

---
<p align="center"><b>© 2026 Murilo Guimarães. Acadêmico de ADS.</b></p>

---